# Conto economico pro forma trimestrale con PROC COMPUTAB

## Sintesi esecutiva

Questo notebook costruisce il conto economico pro forma trimestrale di una banca regionale direttamente dai dati di contabilità mensile usando PROC COMPUTAB, la procedura SAS/ETS per la scrittura di tabelle di report. Instrada gli interessi attivi, gli interessi passivi, i proventi da commissioni e i costi operativi di ciascun mese nella colonna del trimestre di calendario corretta, poi usa blocchi di programmazione di riga per calcolare il margine di interesse, l'utile ante imposte e l'utile netto, e un blocco di colonna per aggregare i quattro trimestri in un totale d'esercizio annuale.

## Fonti dei dati

L'unico dataset sintetico `bank_ledger` simula un esercizio finanziario di voci mensili di bilancio per una banca comunitaria di medie dimensioni. Dodici osservazioni mensili sono generate inline con `call streaminit`/`rand` così che il notebook sia completamente autonomo.

| Variabile | Tipo | Descrizione |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Data di bilancio di fine mese (gen–dic) |
| `loanint`  | Num | Interessi e commissioni maturati sul portafoglio prestiti (migliaia di USD) |
| `secint`   | Num | Interessi maturati sul portafoglio titoli di investimento (migliaia di USD) |
| `depint`   | Num | Interessi pagati sui depositi (migliaia di USD) |
| `borrint`  | Num | Interessi pagati su finanziamenti / anticipazioni FHLB (migliaia di USD) |
| `feeinc`   | Num | Proventi non da interessi (commissioni e spese di servizio) (migliaia di USD) |
| `salaries` | Num | Spese per salari e benefici ai dipendenti (migliaia di USD) |
| `occupancy`| Num | Spese per immobili e attrezzature (migliaia di USD) |
| `othropex` | Num | Altre spese operative non da interessi (migliaia di USD) |
| `provision`| Num | Accantonamento per perdite su crediti (migliaia di USD) |
| `taxrate`  | Num | Aliquota fiscale effettiva applicata all'utile ante imposte |

# Conto economico pro forma trimestrale con PROC COMPUTAB

I team finanziari delle banche trasformano abitualmente una contabilità generale mensile in un **conto economico trimestrale** che mostra il margine di interesse e l'utile netto finale. `PROC COMPUTAB` (SAS/ETS) è concepita appositamente per questo: dispone una tabella programmabile le cui **colonne** sono i periodi di rendicontazione e le cui **righe** sono le voci di bilancio, e consente di calcolare i subtotali con l'ordinaria logica del passo DATA in blocchi di riga e di colonna.

In questo notebook:

1. Generiamo un esercizio finanziario di dati contabili mensili sintetici per una banca comunitaria.
2. Instradiamo ciascun mese nel suo trimestre di calendario e costruiamo un conto economico trimestrale.
3. Calcoliamo il margine di interesse, l'utile ante imposte e l'utile netto in un **blocco di riga**, e aggreghiamo i trimestri in un totale d'esercizio annuale in un **blocco di colonna**.
4. Riutilizziamo la tabella `out=` così che il prospetto calcolato possa alimentare la reportistica a valle.

## Passo 1 — Generare dati contabili mensili sintetici

Simuliamo dodici osservazioni di fine mese. Gli interessi attivi su prestiti e titoli crescono dolcemente nel corso dell'anno, i costi dei depositi e dei finanziamenti scalano con il contesto dei tassi, e i proventi da commissioni, le spese operative e l'accantonamento per perdite su crediti presentano un realistico rumore di mese in mese. `call streaminit` fissa il seme così che i dati siano riproducibili.

In [1]:
DATI bank_ledger;
   CHIAMARE streaminit(20240115);
   FORMATO stmtdate date9.;
   FARE month = 1 FINO_A 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      USCITA;
   FINE;
   MANTENERE stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
ESEGUIRE;

PROCEDURA STAMPARE DATI=bank_ledger noobs ETICHETTA;
   TITOLO 'Synthetic Monthly Ledger — Community Bank FY2024';
   FORMATO loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
ESEGUIRE;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Passo 2 — Costruire il conto economico trimestrale

Il cuore del report è il passo `PROC COMPUTAB` qui sotto.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definisce quattro colonne trimestrali più una colonna d'esercizio annuale.
- Il **blocco di input** senza etichetta imposta la variabile automatica **`_col_`** con `qtr(stmtdate)`, che instrada ciascuna osservazione mensile nella colonna del trimestre corretta. Poiché l'input è trasposto per impostazione predefinita, ogni variabile contabile ricade sulla riga che condivide il suo nome.
- Il **blocco di riga** `inc_stmt:` viene eseguito una volta per colonna e calcola le voci derivate — margine di interesse, spese totali non da interessi, utile ante imposte e utile netto — esattamente come farebbe un contabile.
- Il **blocco di colonna** `total:` viene eseguito una volta per riga e somma i quattro trimestri nella colonna `fy` (esercizio annuale).

Le istruzioni `rows ... / ...` aggiungono titoli, indentazione e linee di regola (`ol` linea sopra, `ul` linea sotto, `dul` doppia linea sotto) così che l'output si legga come un vero prospetto finanziario.

In [2]:
TITOLO 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROCEDURA computab DATI=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / FORMATO=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
ESEGUIRE;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Passo 3 — Riutilizzare il dataset di output di COMPUTAB

Il passo `PROC COMPUTAB` qui sopra ha scritto la sua tabella calcolata in `qtr_income` tramite `out=`. Ogni riga di quel dataset è una voce di bilancio e ogni variabile di colonna (`qtr1`–`qtr4`, `fy`) contiene il valore calcolato, così da poter alimentare la reportistica a valle. Qui sotto stampiamo la colonna d'esercizio annuale aggregata per confermare che le cifre quadrino.

In [3]:
TITOLO 'COMPUTAB Output Dataset — Full-Year Column';

PROCEDURA STAMPARE DATI=qtr_income ETICHETTA noobs;
   VARIABILE _row_ fy;
   FORMATO fy comma12.1;
   ETICHETTA _row_ = 'Line Item' fy = 'Full Year';
ESEGUIRE;

TITOLO;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretazione dei risultati

Il prospetto pro forma si legge dall'alto in basso come il call report regolamentare di una banca: gli interessi attivi totali meno gli interessi passivi totali danno il **margine di interesse** — qui circa \$20,5 milioni per l'anno — il principale motore di reddito della banca. Sottraendo l'**accantonamento per perdite su crediti**, aggiungendo i **proventi da commissioni** e nettando le **spese non da interessi** si ottiene un utile ante imposte di circa \$9,0 milioni, e applicando l'aliquota fiscale effettiva del 21% si produce un **utile netto** vicino a \$7,1 milioni. Il blocco di colonna `total:` somma i quattro trimestri nella colonna d'esercizio annuale, quindi i totali annuali si riconciliano per costruzione con la somma dei trimestri — confermato nel dataset `out=`, dove il `netinc` d'esercizio annuale di 7.081,6 è uguale alla somma delle quattro cifre trimestrali.

Poiché tutto è calcolato all'interno dei blocchi programmabili di riga e colonna di `PROC COMPUTAB`, sostituire una contabilità mensile reale non richiede alcuna modifica alla logica del report — cambia solo il dataset di input. Il dataset `out=` rende poi il prospetto calcolato disponibile per dashboard, analisi di tendenza o automazione del pacchetto per il consiglio direttivo.